# 07. Memory Layouts, Strides & Contiguity (5+ Years Interview Guide)
Exhaustive revision guide to C_CONTIGUOUS vs F_CONTIGUOUS memory layouts, stride byte mathematics, and enforcing contiguity with np.ascontiguousarray() on transaction matrices.

### Key 5-Year Interview Concepts Covered:
- **Layout Inspection (`arr.flags`)**: Checking `C_CONTIGUOUS` (row-major) vs `F_CONTIGUOUS` (column-major).
- **Stride Byte Navigation (`arr.strides`)**: Inspecting bytes skipped along each axis.
- **Transposition Stride Swapping**: Understanding zero-copy transpose pointer mechanics.
- **Enforcing Contiguity (`np.ascontiguousarray`)**: Converting non-contiguous strided arrays back into sequential RAM.

This interactive revision guide loads and operates directly on `data/raw_transactions.csv` using dedicated cells per method.

In [ ]:
# Setup imports & dataset loading from raw_transactions.csv
import numpy as np
import pandas as pd
import sys
import time
import os

# Load raw transactions and extract aligned NumPy numeric arrays
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
raw_df = pd.read_csv(csv_path)
clean_raw = raw_df.dropna(subset=['transaction_amount', 'is_fraud', 'account_age_months']).reset_index(drop=True)
amounts = clean_raw['transaction_amount'].to_numpy(dtype=np.float64)
fraud_flags = clean_raw['is_fraud'].to_numpy(dtype=np.int8)
account_ages = clean_raw['account_age_months'].to_numpy(dtype=np.float32)

print(f"NumPy Version: {np.__version__}")
print(f"Loaded from {csv_path} ({len(amounts)} clean aligned rows):")
print(f"- amounts array: shape {amounts.shape}, dtype {amounts.dtype}")
print(f"- fraud_flags array: shape {fraud_flags.shape}, dtype {fraud_flags.dtype}")
print(f"- account_ages array: shape {account_ages.shape}, dtype {account_ages.dtype}")

### Layout Inspection with `arr.flags`
**Explanation**: Checks whether a 2D transaction matrix is row-major (C-order) or column-major (Fortran-order).

**Syntax**: `tx_mat.flags['C_CONTIGUOUS']`

In [ ]:
tx_mat = np.column_stack([amounts[:1000], account_ages[:1000]])
print('Transaction Matrix C_CONTIGUOUS:', tx_mat.flags.c_contiguous)

### Stride Byte Navigation: `arr.strides`
**Explanation**: Inspects the byte-step stride tuple for navigating rows and columns in the transaction matrix.

**Syntax**: `tx_mat.strides`

In [ ]:
print('Transaction Matrix Shape:', tx_mat.shape)
print('Transaction Matrix Strides (bytes):', tx_mat.strides)

### Zero-Copy Transposition Stride Swapping
**Explanation**: Transposing the transaction matrix swaps strides in $O(1)$ time without copying memory.

**Syntax**: `tx_mat.T.strides`

In [ ]:
tx_transposed = tx_mat.T
print('Transposed Shape:', tx_transposed.shape)
print('Transposed Strides:', tx_transposed.strides)
print('Is Transposed Matrix C_CONTIGUOUS?:', tx_transposed.flags.c_contiguous)

### Enforcing Contiguity with `np.ascontiguousarray()`
**Explanation**: Converts the transposed non-contiguous matrix back into a contiguous C-memory layout for Cython/C++ extensions.

**Syntax**: `np.ascontiguousarray(tx_transposed)`

In [ ]:
contig_tx = np.ascontiguousarray(tx_transposed)
print('Re-enforced Contiguity C_CONTIGUOUS:', contig_tx.flags.c_contiguous)

## Section: Senior Fintech Interview Scenarios (5+ Years Experience)

### Q1: Row-Wise vs Column-Wise Traversal Performance Benchmark
**Explanation**: Benchmark traversal speed across contiguous rows versus non-contiguous columns on transaction matrix.

**Syntax**: `tx_mat.sum(axis=1)` vs `tx_mat.sum(axis=0)`

In [ ]:
big_mat = np.tile(tx_mat, (10, 1))
t0 = time.perf_counter()
big_mat.sum(axis=1)
t_row = time.perf_counter() - t0

t0 = time.perf_counter()
big_mat.sum(axis=0)
t_col = time.perf_counter() - t0
print(f'Row-wise (Cache-friendly): {t_row*1000:.2f} ms')
print(f'Col-wise: {t_col*1000:.2f} ms')